[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.2_eks_kserve/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.2_eks_kserve/lab.ipynb)

# 7.2 KServe on EKS - Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.2_eks_kserve/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud.coreweavex.com/hub/user-redirect/lab/tree/llm-inference-at-scale/content/08_serving/07.2_eks_kserve/lab.ipynb)

This lab explores KServe InferenceService configuration, autoscaling simulation, and cold start analysis for LLM workloads on EKS.

In [ ]:
# Install dependencies for KServe manifest generation and analysis
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyyaml', 'matplotlib', 'numpy'], check=True)

# Import core libraries for simulation and visualization
import yaml  # YAML generation for Kubernetes manifests
import numpy as np  # Numerical computation for scaling simulation
import matplotlib.pyplot as plt  # Plotting autoscaling behavior

## Exercise 1: Generate KServe InferenceService Manifest

Build a production-ready InferenceService YAML for an LLM endpoint.

In [ ]:
def generate_kserve_manifest(
    model_name: str,  # Name for the InferenceService resource
    model_uri: str,  # S3 path to model weights
    gpu_count: int,  # Number of GPUs per replica
    min_replicas: int,  # Minimum pod count (>= 1 avoids cold starts)
    max_replicas: int,  # Maximum pod count for autoscaling ceiling
    instance_memory: str,  # Memory request per pod (e.g. '48Gi')
) -> dict:
    """Generate a KServe InferenceService manifest for LLM serving."""
    # Build the InferenceService custom resource structure
    manifest = {
        'apiVersion': 'serving.kserve.io/v1beta1',  # KServe API version
        'kind': 'InferenceService',  # Resource type
        'metadata': {
            'name': model_name,  # Unique name within namespace
            'annotations': {
                # Set minimum replicas to avoid cold starts
                'autoscaling.knative.dev/minScale': str(min_replicas),
                # Set maximum replicas as cost ceiling
                'autoscaling.knative.dev/maxScale': str(max_replicas),
            }
        },
        'spec': {
            'predictor': {
                'model': {
                    'modelFormat': {'name': 'vllm'},  # Use vLLM as serving engine
                    'storageUri': model_uri,  # S3 location of model weights
                    'resources': {
                        'limits': {
                            'nvidia.com/gpu': str(gpu_count),  # GPU allocation
                            'memory': instance_memory,  # Memory ceiling
                        },
                        'requests': {
                            'nvidia.com/gpu': str(gpu_count),  # Guaranteed GPU
                            'memory': instance_memory,  # Guaranteed memory
                        }
                    }
                }
            }
        }
    }
    return manifest

# Generate manifest for Llama-3.1-8B on single A10G GPU
llama_manifest = generate_kserve_manifest(
    model_name='llama-3-8b-vllm',  # Endpoint name
    model_uri='s3://my-models/llama-3.1-8b/',  # Model weight location
    gpu_count=1,  # Single GPU for 8B model
    min_replicas=1,  # Always keep 1 pod warm
    max_replicas=8,  # Scale up to 8 replicas under load
    instance_memory='24Gi',  # A10G has 24GB VRAM
)

# Print the generated YAML manifest
print(yaml.dump(llama_manifest, default_flow_style=False))

## Exercise 2: Autoscaling Simulation

Simulate how KServe autoscaling responds to traffic spikes with LLM-specific latency characteristics.

In [ ]:
def simulate_autoscaling(
    traffic_pattern: np.ndarray,  # Requests per second over time
    capacity_per_pod: int,  # Max concurrent requests per pod
    scale_up_threshold: float,  # Queue depth ratio triggering scale-up
    scale_down_cooldown: int,  # Seconds before scale-down allowed
    cold_start_seconds: int,  # Time for new pod to become ready
    min_replicas: int,  # Floor for autoscaler
    max_replicas: int,  # Ceiling for autoscaler
) -> tuple:
    """Simulate pod scaling decisions over a traffic pattern."""
    # Initialize tracking arrays
    n_steps = len(traffic_pattern)  # Number of time steps
    replicas = np.zeros(n_steps)  # Active replica count per step
    queue_depth = np.zeros(n_steps)  # Pending requests per step
    replicas[0] = min_replicas  # Start at minimum
    last_scale_down = -scale_down_cooldown  # Allow immediate first scale
    pending_pods = []  # (ready_at_step, count) for pods being started

    for t in range(1, n_steps):
        # Check if any pending pods have finished cold start
        ready_now = sum(c for (ready_at, c) in pending_pods if ready_at <= t)
        # Remove pods that have become ready
        pending_pods = [(r, c) for (r, c) in pending_pods if r > t]
        # Update active replica count with newly ready pods
        active = replicas[t-1] + ready_now

        # Calculate current capacity and queue pressure
        total_capacity = active * capacity_per_pod  # Max requests handleable
        overflow = max(0, traffic_pattern[t] - total_capacity)  # Excess requests
        queue_depth[t] = overflow  # Record queue pressure

        # Scale-up decision: queue exceeds threshold
        if overflow > scale_up_threshold * capacity_per_pod and active < max_replicas:
            # Calculate how many pods needed to absorb overflow
            needed = int(np.ceil(overflow / capacity_per_pod))
            # Cap at max_replicas
            needed = min(needed, max_replicas - int(active))
            # Schedule pods with cold start delay
            pending_pods.append((t + cold_start_seconds, needed))

        # Scale-down decision: excess capacity and cooldown elapsed
        elif overflow == 0 and active > min_replicas and (t - last_scale_down) > scale_down_cooldown:
            # Remove one pod at a time for graceful drain
            active = max(min_replicas, active - 1)
            last_scale_down = t  # Reset cooldown timer

        replicas[t] = active  # Record final replica count

    return replicas, queue_depth

# Simulate a traffic spike scenario (600 seconds)
np.random.seed(42)  # Reproducible results
time_steps = 600  # 10 minutes of traffic
# Create traffic pattern: baseline 10 RPS, spike to 80 RPS at t=200
traffic = np.concatenate([
    np.full(200, 10),  # Steady low traffic
    np.linspace(10, 80, 50),  # Ramp up over 50 seconds
    np.full(200, 80),  # Sustained high traffic
    np.linspace(80, 10, 50),  # Ramp down
    np.full(100, 10),  # Return to baseline
])

# Run autoscaling simulation with LLM-realistic parameters
replicas, queue = simulate_autoscaling(
    traffic_pattern=traffic,  # Our traffic scenario
    capacity_per_pod=15,  # Each vLLM pod handles ~15 concurrent requests
    scale_up_threshold=0.5,  # Scale when queue > 50% of pod capacity
    scale_down_cooldown=60,  # Wait 60s before removing a pod
    cold_start_seconds=45,  # 45s to load model weights
    min_replicas=1,  # Always keep 1 pod
    max_replicas=8,  # Cost ceiling
)

# Plot the autoscaling behavior
fig_2, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Top: Traffic pattern
ax1.plot(traffic, color='#2563eb', linewidth=1.5)  # Blue line for traffic
ax1.set_ylabel('Requests/sec')  # Y-axis label
ax1.set_title('KServe Autoscaling Simulation for LLM Endpoint')  # Chart title
ax1.fill_between(range(len(traffic)), traffic, alpha=0.2, color='#2563eb')  # Fill area

# Middle: Replica count
ax2.step(range(len(replicas)), replicas, color='#166534', linewidth=2)  # Step plot for replicas
ax2.set_ylabel('Active Replicas')  # Y-axis label
ax2.axhline(y=1, color='#991b1b', linestyle='--', label='Min replicas')  # Min line
ax2.legend()  # Show legend

# Bottom: Queue depth (user-visible wait)
ax3.fill_between(range(len(queue)), queue, color='#991b1b', alpha=0.4)  # Red fill for queue
ax3.set_ylabel('Queue Depth')  # Y-axis label
ax3.set_xlabel('Time (seconds)')  # X-axis label

plt.tight_layout()  # Remove whitespace between subplots
plt.show()  # Render the figure

## Exercise 3: Cold Start Impact Analysis

Compare user-visible latency impact of different cold start strategies.

In [ ]:
def cold_start_comparison():
    """Compare cold start times across storage and caching strategies."""
    # Define strategies with their cold start components (seconds)
    strategies = {
        'S3 Standard': {
            'download': 90,  # 14GB model from S3 at ~155 MB/s
            'container_start': 15,  # Pull and start container
            'weight_load': 20,  # Load weights into GPU memory
        },
        'S3 Express': {
            'download': 12,  # 14GB at ~1.1 GB/s (Express One Zone)
            'container_start': 15,  # Same container overhead
            'weight_load': 20,  # Same GPU load time
        },
        'FSx Lustre': {
            'download': 2,  # 14GB at ~10 GB/s (parallel filesystem)
            'container_start': 15,  # Same container overhead
            'weight_load': 20,  # Same GPU load time
        },
        'Pre-pulled Image': {
            'download': 0,  # Weights baked into container image
            'container_start': 5,  # Image already cached on node
            'weight_load': 20,  # Still need to load into GPU
        },
    }

    # Create horizontal bar chart comparing strategies
    fig_3, ax_3 = plt.subplots(figsize=(10, 5))
    labels = list(strategies.keys())  # Strategy names for y-axis
    # Extract component times for stacked bars
    downloads = [s['download'] for s in strategies.values()]  # Download time
    containers = [s['container_start'] for s in strategies.values()]  # Container time
    loads = [s['weight_load'] for s in strategies.values()]  # GPU load time

    y_pos = range(len(labels))  # Y positions for bars
    # Plot stacked horizontal bars for each component
    ax_3.barh(y_pos, downloads, color='#dbeafe', edgecolor='#000', label='Download')
    ax_3.barh(y_pos, containers, left=downloads, color='#fef3c7', edgecolor='#000', label='Container Start')
    # Calculate left offset for weight loading bars
    left_offset = [d + c for d, c in zip(downloads, containers)]
    ax_3.barh(y_pos, loads, left=left_offset, color='#dcfce7', edgecolor='#000', label='Weight Load')

    # Add total time labels at end of each bar
    for i, (d, c, l) in enumerate(zip(downloads, containers, loads)):
        total = d + c + l  # Sum all components
        ax_3.text(total + 2, i, f'{total}s', va='center', fontsize=10)  # Label

    ax_3.set_yticks(y_pos)  # Set y-axis tick positions
    ax_3.set_yticklabels(labels)  # Set y-axis labels
    ax_3.set_xlabel('Cold Start Time (seconds)')  # X-axis label
    ax_3.set_title('KServe Cold Start: Impact of Storage Strategy (14B model)')  # Title
    ax_3.legend(loc='lower right')  # Legend position
    plt.tight_layout()  # Clean layout
    plt.show()  # Render

# Run the cold start comparison
cold_start_comparison()